# 01 — Latest-file Bronze ingestion

Load the current export into `bronze` as raw Delta tables and record run,
table, and load-control metrics in the shared `monitoring` schema.

In [ ]:
ROOT_PATH = "Files/wmpp-production-data-export-birmingham/latest"
BRONZE_SCHEMA = "bronze"
TABLE_PREFIX = ""
TEXT_QUALIFIER = '"'
REBUILD = False
FRAMEWORK_PATH = "Files/deprecated_wmpp_files/framework.csv"

In [ ]:
import os, re, uuid
from datetime import datetime
from notebookutils import mssparkutils
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()

def clean_name(file_name):
    stem = os.path.splitext(os.path.basename(file_name.rstrip("/")))[0]
    return re.sub(r"[^a-zA-Z0-9_]+", "_", stem).strip("_").lower()

def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS monitoring")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_pipeline_run (
  run_id STRING, pipeline_name STRING, layer STRING, source_kind STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, status STRING,
  tables_succeeded INT, tables_failed INT, rows_read BIGINT, rows_written BIGINT,
  error_message STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_table_load_metric (
  run_id STRING, layer STRING, source_kind STRING, source_object STRING,
  target_object STRING, rows_read BIGINT, rows_written BIGINT,
  duplicate_key_count BIGINT, null_primary_key_count BIGINT,
  recorded_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_bronze_load_control (
  run_id STRING, source_path STRING, source_file STRING, target_table STRING,
  load_mode STRING, status STRING, rows_read BIGINT, rows_written BIGINT,
  column_count INT, started_at TIMESTAMP, ended_at TIMESTAMP, error_message STRING
) USING DELTA
""")
append_rows("monitoring.cfg_pipeline_run", [(RUN_ID, "01_bronze_get_latest", "BRONZE", "LATEST",
    STARTED_AT, None, "RUNNING", 0, 0, 0, 0, None)],
    "run_id string,pipeline_name string,layer string,source_kind string,started_at timestamp,ended_at timestamp,status string,tables_succeeded int,tables_failed int,rows_read long,rows_written long,error_message string")

In [ ]:
items = list(mssparkutils.fs.ls(ROOT_PATH))
if mssparkutils.fs.exists(FRAMEWORK_PATH):
    items.append(type("FileInfo", (), {"path": FRAMEWORK_PATH, "name": "framework.csv"})())

control_schema = "run_id string,source_path string,source_file string,target_table string,load_mode string,status string,rows_read long,rows_written long,column_count int,started_at timestamp,ended_at timestamp,error_message string"
metric_schema = "run_id string,layer string,source_kind string,source_object string,target_object string,rows_read long,rows_written long,duplicate_key_count long,null_primary_key_count long,recorded_at timestamp"
ok = failed = total_read = total_written = 0
errors = []

for item in items:
    source_path = item.path
    source_file = os.path.basename(source_path.rstrip("/"))
    table_name = TABLE_PREFIX + clean_name(source_file)
    target = f"{BRONZE_SCHEMA}.{table_name}"
    started = datetime.utcnow()
    try:
        if REBUILD:
            spark.sql(f"DROP TABLE IF EXISTS {target}")
        if source_file.lower().endswith(".parquet"):
            frame = spark.read.format("parquet").load(source_path)
        elif source_file.lower().endswith(".csv"):
            frame = (spark.read.format("csv").option("header", "true").option("inferSchema", "false")
                .option("mode", "PERMISSIVE").option("quote", TEXT_QUALIFIER)
                .option("escape", TEXT_QUALIFIER).option("multiLine", "true").load(source_path))
        else:
            continue

        frame = (frame.withColumn("_source_file_path", F.lit(source_path))
            .withColumn("_bronze_run_id", F.lit(RUN_ID))
            .withColumn("_bronze_load_ts", F.current_timestamp()))
        row_count = frame.count()
        frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
        append_rows("monitoring.cfg_bronze_load_control", [(RUN_ID, source_path, source_file, target,
            "OVERWRITE", "SUCCESS", row_count, row_count, len(frame.columns), started, datetime.utcnow(), None)], control_schema)
        append_rows("monitoring.cfg_table_load_metric", [(RUN_ID, "BRONZE", "LATEST", source_path,
            target, row_count, row_count, None, None, datetime.utcnow())], metric_schema)
        ok += 1; total_read += row_count; total_written += row_count
    except Exception as exc:
        message = str(exc)[:2000]
        errors.append(f"{source_file}: {message}")
        failed += 1
        append_rows("monitoring.cfg_bronze_load_control", [(RUN_ID, source_path, source_file, target,
            "OVERWRITE", "FAILED", 0, 0, 0, started, datetime.utcnow(), message)], control_schema)

status = "FAILED" if errors else "SUCCESS"
error_text = " | ".join(errors)[:4000] if errors else None
error_sql = "NULL" if error_text is None else "'" + error_text.replace("'", "''") + "'"
spark.sql(f"""UPDATE monitoring.cfg_pipeline_run SET ended_at=current_timestamp(), status='{status}',
tables_succeeded={ok}, tables_failed={failed}, rows_read={total_read}, rows_written={total_written},
error_message={error_sql} WHERE run_id='{RUN_ID}'""")
if errors:
    raise RuntimeError(error_text)
print(f"Bronze run {RUN_ID}: {ok} tables, {total_written:,} rows")